In [ ]:
# --- Install Playwright and dependencies ---
!pip install playwright

# Install the browsers (Chromium, Firefox, WebKit)
!playwright install

# (Optional) Install additional system dependencies for headless browsers
!playwright install-deps

!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 MB 16.0 MB/s eta 0:00:00
173.7 MiB [] 0% 0.0s173.7 MiB [] 0% 4.7s173.7 MiB [] 1% 2.8s173.7 MiB [] 1% 2.4s173.7 MiB [] 2% 2.1s173.7 MiB [] 4% 1.9s173.7 MiB [] 5% 2.0s173.7 MiB [] 5% 2.4s173.7 MiB [] 6% 2.5s173.7 MiB [] 6% 2.6s173.7 MiB [] 7% 2.6s173.7 MiB [] 8% 2.6s173.7 MiB [] 9% 2.6s173.7 MiB [] 10% 2.5s173.7 MiB [] 11% 2.4s173.7 MiB [] 12% 2.2s173.7 MiB [] 13% 2.2s173.7 MiB [] 14% 2.1s173.7 MiB [] 15% 2.0s173.7 MiB [] 17% 1.9s173.7 MiB [] 18% 1.8s173.7 MiB [] 19% 1.7s173.7 MiB [] 20% 1.7s173.7 MiB [] 21% 1.7s173.7 MiB [] 22% 1.7s173.7 MiB [] 24% 1.6s173.7 MiB [] 25% 1.5s173.7 MiB [] 26% 1.5s173.7 MiB [] 26% 1.6s173.7 MiB [] 28% 1.5s173.7 MiB [] 29% 1.5s173.7 MiB [] 31% 1.4s173.7 MiB [] 32% 1.3s173.7 MiB [] 33% 1.3s173.7 MiB [] 35% 1.2s173.7 MiB [] 36% 1.2s173.7 MiB [] 38% 1.1s173.7 MiB [] 39% 1.1s173.7 MiB [] 40% 1.1s173.7 MiB [] 42% 1.0s173.7 MiB [] 43% 1.0s173.7 MiB [] 45% 1.0s173.7 MiB [] 46% 0.9s173.7 MiB [] 47% 0.9s173.7 MiB 

In [ ]:
import asyncio
from playwright.async_api import async_playwright

async def main():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto("https://example.com")
        print("Title:", await page.title())
        print("URL:", page.url)
        await page.screenshot(path="example.png")
        await browser.close()

# Run inside the notebook's async event loop
await main()


Title: Example Domain
URL: https://example.com/


In [ ]:
import asyncio
from datetime import date, timedelta
from playwright.async_api import async_playwright
from reportlab.pdfgen import canvas
import re
import os

JOTFORM_URL = "https://eel.jotform.com/252244495892972"

def make_test_pdf(filename="invoice.pdf"):
    c = canvas.Canvas(filename)
    c.drawString(100, 750, "Test Invoice PDF – Automated Upload")
    c.save()

async def run_test_form():
    make_test_pdf("invoice.pdf")

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        # ===== STAGE 1: INPUTTER STAGE =====
        print("\n===== STAGE 1: INPUTTER STAGE =====")
        await page.goto(JOTFORM_URL)
        print("Navigated to form")

        # 1. Company / Division
        await page.select_option("#input_123", label="Edmundson Electrical")
        print("Completed: Company / Division")

        # 2. Location Number
        await page.fill("#input_124", "331")
        print("Completed: Location Number")

        # 3. Raised By
        await page.fill("#input_131", "Automated Test")
        print("Completed: Raised By")

        # 4. Payment Request Date - confirm it has a value
        date_value = await page.input_value("#lite_mode_130")
        if date_value:
            print(f"Completed: Payment Request Date has value: {date_value}")
        else:
            print("Warning: Payment Request Date is empty")

        # 5. Payment Request Type
        await page.select_option("#input_543", label="Sponsorship/Charitable Donation")
        print("Completed: Payment Request Type")

        # 6. Description
        await page.fill("#input_701", "Test Description")
        print("Completed: Description")

        # 7. Payee
        await page.fill("#input_547", "Test Auto")
        print("Completed: Payee")

        # 8. Do you have an invoice? -> Yes (click the label instead)
        await page.click("#label_input_604_0")
        print("Completed: Do you have an invoice? (Yes)")

        # 9. Upload PDF file (valid format)
        await page.set_input_files("#input_558", "invoice.pdf")
        print("Completed: Upload PDF file")

        # 10. Invoice Number
        await page.fill("#input_562", "Invoice123")
        print("Completed: Invoice Number")

        # 11. Invoice Date (yesterday)
        yesterday = (date.today() - timedelta(days=1)).strftime("%d%m%Y")
        await page.fill("#lite_mode_566", yesterday)
        print("Completed: Invoice Date")

        # 12. Bank Account Details on invoice? -> Yes (click the label instead)
        await page.click("#label_input_607_0")
        print("Completed: Bank Account Details on invoice? (Yes)")

        # 13. Value
        await page.fill("#input_844", "10000")
        print("Completed: Value")

        # 14. Profit Centre Manager Approver (iframe handling)
        frame = None
        for f in page.frames:
            if "ADDropdown" in (f.name or "") or "ADDropdown" in f.url:
                frame = f
                break

        if frame:
            dropdown = await frame.query_selector("#input_ADDropdown")
            options = await dropdown.query_selector_all("option")
            if len(options) > 1:
                second_value = await options[1].get_attribute("value")
                await frame.select_option("#input_ADDropdown", value=second_value)
                approver_text = await options[1].text_content()
                print(f"Completed: Profit Centre Manager Approver - Selected: {approver_text}")
            else:
                print("Warning: No second option found in approver dropdown")
        else:
            print("Warning: Approver dropdown iframe not found.")

        # 15. Profit Centre Manager Email
        await page.fill("#input_195", "mustapha.jobe@digiblu.com")
        print("Completed: Profit Centre Manager Email")

        # 16. Submit and verify redirect
        initial_url = page.url
        await page.click("#input_98")
        print("Completed: Submit button clicked")

        # Wait for potential redirect
        await page.wait_for_timeout(3000)

        # Check if redirected
        current_url = page.url
        if current_url == initial_url:
            print("Warning: No redirect detected. Retrying submit...")
            await page.click("#input_98")
            await page.wait_for_timeout(3000)

            # Check again
            current_url = page.url
            if current_url == initial_url:
                print("Error: Still no redirect after retry")
                await browser.close()
                return
            else:
                print(f"Success: Redirected to {current_url}")
        else:
            print(f"Success: Redirected to {current_url}")

        # Extract EFS Ref from the thank you page
        page_content = await page.content()
        efs_ref_match = re.search(r'EFS Ref:\s*(\S+)', page_content)

        if not efs_ref_match:
            print("Error: Could not find EFS Ref on thank you page")
            await browser.close()
            return

        efs_ref = efs_ref_match.group(1)
        print(f"Found EFS Ref: {efs_ref}")

        # Create folder with EFS Ref name
        folder_name = efs_ref
        if not os.path.exists(folder_name):
            os.makedirs(folder_name)
            print(f"Created folder: {folder_name}")
        else:
            print(f"Folder already exists: {folder_name}")

        # Save Stage 1 screenshot
        await page.screenshot(path=f"{folder_name}/stage1_submitted.png")
        print("Stage 1 screenshot saved.")

        # Extract the edit link from the page
        edit_link_match = re.search(r'https://eel\.jotform\.com/edit/[^\s\'"<>]+', page_content)

        if not edit_link_match:
            print("Error: Could not find edit link for PCM Stage")
            await browser.close()
            return

        pcm_stage_url = edit_link_match.group(0)
        print(f"Found PCM Stage edit link: {pcm_stage_url}")

        # Wait 10 seconds for background workflows
        print("Waiting 10 seconds for background workflows...")
        await page.wait_for_timeout(10000)

        # ===== PCM STAGE =====
        print("\n===== PCM STAGE =====")
        await page.goto(pcm_stage_url)
        print("Navigated to PCM Stage form")

        # 1. Profit Centre Manager Approve or Reject -> Approve
        await page.click("#label_input_203_0")
        print("Completed: Profit Centre Manager Approve or Reject (Approve)")

        # 2. Regional Director Email for Testing
        await page.fill("#input_244", "mustapha.jobe@digiblu.com")
        print("Completed: Regional Director Email for Testing")

        # Submit PCM Stage
        initial_url = page.url
        await page.click("#input_98")
        print("Completed: PCM Stage Submit button clicked")

        await page.wait_for_timeout(3000)

        # Check if redirected
        current_url = page.url
        if current_url == initial_url:
            print("Warning: No redirect detected. Retrying submit...")
            await page.click("#input_98")
            await page.wait_for_timeout(3000)

            current_url = page.url
            if current_url == initial_url:
                print("Error: Still no redirect after retry")
                await browser.close()
                return
            else:
                print(f"Success: Redirected to {current_url}")
        else:
            print(f"Success: Redirected to {current_url}")

        await page.screenshot(path=f"{folder_name}/pcm_stage_submitted.png")
        print("PCM Stage screenshot saved.")

        # Extract the edit link for RD Stage
        page_content = await page.content()
        edit_link_match = re.search(r'https://eel\.jotform\.com/edit/[^\s\'"<>]+', page_content)

        if not edit_link_match:
            print("Error: Could not find edit link for RD Stage")
            await browser.close()
            return

        rd_stage_url = edit_link_match.group(0)
        print(f"Found RD Stage edit link: {rd_stage_url}")

        # Wait 10 seconds for background workflows
        print("Waiting 10 seconds for background workflows...")
        await page.wait_for_timeout(10000)

        # ===== RD STAGE =====
        print("\n===== RD STAGE =====")
        await page.goto(rd_stage_url)
        print("Navigated to RD Stage form")

        # 1. Regional Director Approve or Reject -> Approve
        await page.click("#label_input_249_0")
        print("Completed: Regional Director Approve or Reject (Approve)")

        # Submit RD Stage
        initial_url = page.url
        await page.click("#input_98")
        print("Completed: RD Stage Submit button clicked")

        await page.wait_for_timeout(3000)

        # Check if redirected
        current_url = page.url
        if current_url == initial_url:
            print("Warning: No redirect detected. Retrying submit...")
            await page.click("#input_98")
            await page.wait_for_timeout(3000)

            current_url = page.url
            if current_url == initial_url:
                print("Error: Still no redirect after retry")
            else:
                print(f"Success: Redirected to {current_url}")
        else:
            print(f"Success: Redirected to {current_url}")

        await page.screenshot(path=f"{folder_name}/rd_stage_submitted.png")
        print("RD Stage screenshot saved.")

        print("\n===== ALL STAGES COMPLETED =====")
        print(f"All screenshots saved in folder: {folder_name}")
        await browser.close()

await run_test_form()